# File 07 — Dashboard Export
**EHSL-ML+ | Disertasi Nanan Soekarna**

File terakhir. Tugasnya sederhana tapi penting: **mengumpulkan semua output** dari File 02–06 dan menyusunnya jadi satu struktur data (`dashboard_data.json`) yang siap divisualisasikan, mengikuti arsitektur **5-layer dashboard** dari proposal (Tabel 4.5).

| Layer Dashboard | Isi | Sumber |
|---|---|---|
| Layer 1 — Index Score Panel | Skor VfV/FCNC/ID/MAS/ECI/NREI, kategori, per aktor | File 02 |
| Layer 2 — Feature Engineering Panel | Temporal pattern, ECV consistency, interaction | File 04 |
| Layer 3 — ML Analytics Panel | Cluster, klasifikasi, anomali | File 05 |
| Layer 4 — Explainability Panel | SHAP top features | File 06 |
| Layer 5 — ERM Score Panel | ERM per aktor/stage, Reward-Penalty | File 06 |

File ini tidak membuat model baru — cuma **merangkai** yang sudah ada jadi format terstruktur.

In [1]:
import numpy as np
import pandas as pd
import json

def clean(obj):
    """Ubah NaN & numpy types jadi JSON-serializable."""
    if isinstance(obj, dict): return {k: clean(v) for k, v in obj.items()}
    if isinstance(obj, list): return [clean(x) for x in obj]
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(obj) else round(float(obj), 3)
    if np.isscalar(obj) and pd.isna(obj): return None
    return obj

## 1. Muat Semua Output dari File Sebelumnya

In [2]:
idx = pd.read_csv("data_index_respondent.csv")
idx_actor = pd.read_csv("data_index_actor.csv")
fm = pd.read_csv("feature_matrix.csv")
ml = pd.read_csv("ml_results.csv")
erm = pd.read_csv("erm_scores.csv")
erm_actor = pd.read_csv("erm_by_actor.csv")
erm_stage = pd.read_csv("erm_by_stage.csv")
shap_imp = pd.read_csv("shap_global_importance.csv")
model_cmp = pd.read_csv("model_comparison.csv")
sens = pd.read_csv("sensitivity_summary.csv")
print("Semua output dari File 02-06 dimuat.")

dashboard = {}

Semua output dari File 02-06 dimuat.


## 2. Meta Information

Catatan penting yang WAJIB ada: data ini dummy. Supaya siapapun yang membuka dashboard tidak salah kira ini hasil riil.

In [3]:
dashboard["meta"] = {
    "framework": "EHSL-ML+ (Ethical-Governance Survey-driven and Human-supervised Learning Plus)",
    "n_respondents": int(len(idx)),
    "n_actor_observations": int(len(idx_actor)),
    "data_type": "DUMMY (simulasi) - BUKAN data riil",
    "note": "Semua angka bersifat ilustratif untuk demonstrasi pipeline.",
}
print(json.dumps(dashboard["meta"], indent=2, ensure_ascii=False))

{
  "framework": "EHSL-ML+ (Ethical-Governance Survey-driven and Human-supervised Learning Plus)",
  "n_respondents": 199,
  "n_actor_observations": 896,
  "data_type": "DUMMY (simulasi) - BUKAN data riil",
  "note": "Semua angka bersifat ilustratif untuk demonstrasi pipeline."
}


## 3. Layer 1 — Index Score Panel

In [4]:
index_cols = ["VfV_overall","FCNC_overall","ID_overall","MAS_overall","ECI_overall","NREI_overall"]
dashboard["layer1_index_scores"] = {
    "national_means": clean({c.replace("_overall",""): idx[c].mean() for c in index_cols}),
    "nrei_category_distribution": clean(idx["NREI_overall_cat3"].value_counts().to_dict()),
    "actor_specific_means": clean(
        idx_actor.groupby("actor_category")[["VfV_norm","FCNC_norm","ID_norm","MAS_actor","ECI_actor","NREI_actor"]]
        .mean().round(1).reset_index().to_dict(orient="records")),
}
print("Layer 1 - rata-rata nasional:")
print(json.dumps(dashboard["layer1_index_scores"]["national_means"], indent=2))

Layer 1 - rata-rata nasional:
{
  "VfV": 48.853,
  "FCNC": 49.274,
  "ID": 49.516,
  "MAS": 49.076,
  "ECI": 49.37,
  "NREI": 49.323
}


## 4. Layer 2 — Feature Engineering Panel

Temporal pattern (perubahan persepsi antar tahap t1-t4) dan ECV consistency per aktor — untuk heatmap/chart dashboard.

In [5]:
stage_means = {}
for construct in ["VfV","FCNC","ID"]:
    stage_means[construct] = clean({f"t{i}": fm[f"T_{construct}_t{i}"].mean() for i in range(1,5)})

ecv_cols = [c for c in fm.columns if c.startswith("ECV_") and c.endswith("_Consistency")]
dashboard["layer2_feature_engineering"] = {
    "temporal_pattern_by_stage": stage_means,
    "mean_consistency_by_actor": clean({
        c.replace("ECV_","").replace("_Consistency",""): fm[c].mean() for c in ecv_cols}),
    "interaction_feature_means": clean({
        c: fm[c].mean() for c in ["IF_VfV_FCNC","IF_FCNC_ID","IF_VfV_ID","IF_ID_Stage"]}),
}
print("Layer 2 - temporal pattern VfV per tahap:")
print(json.dumps(stage_means["VfV"], indent=2))

Layer 2 - temporal pattern VfV per tahap:
{
  "t1": 48.927,
  "t2": 48.806,
  "t3": 48.62,
  "t4": 49.196
}


## 5. Layer 3 — ML Analytics Panel

In [6]:
dashboard["layer3_ml_analytics"] = {
    "model_comparison": clean(model_cmp.to_dict(orient="records")),
    "cluster_distribution": clean(ml["cluster_kmeans"].value_counts().sort_index().to_dict()),
    "n_anomalies": {
        "isolation_forest": int(ml["anomaly_iso"].sum()),
        "lof": int(ml["anomaly_lof"].sum()),
        "agreement_both": int(((ml["anomaly_iso"]==1)&(ml["anomaly_lof"]==1)).sum()),
    },
    "sensitivity_check": clean(sens.to_dict(orient="records")[0]),
}
print("Layer 3 - perbandingan model:")
print(json.dumps(dashboard["layer3_ml_analytics"]["model_comparison"], indent=2))

Layer 3 - perbandingan model:
[
  {
    "model": "Random Forest",
    "accuracy": 0.875,
    "macro_f1": 0.8818752307124399
  },
  {
    "model": "XGBoost",
    "accuracy": 0.825,
    "macro_f1": 0.8326051209772141
  },
  {
    "model": "Logistic Regression",
    "accuracy": 0.825,
    "macro_f1": 0.8247046880767811
  }
]


## 6. Layer 4 — Explainability Panel

In [7]:
dashboard["layer4_explainability"] = {
    "top_features_shap": clean(shap_imp.head(10).to_dict(orient="records")),
}
print("Layer 4 - top 5 fitur (SHAP):")
for f in dashboard["layer4_explainability"]["top_features_shap"][:5]:
    print(f"  {f['feature']}: {f['mean_abs_shap']}")

Layer 4 - top 5 fitur (SHAP):
  T_ID_t3: 0.0239180992271545
  ECV_Investigator_ID: 0.0239104770731896
  ECV_CorrectionalOfficer_FCNC: 0.0235381092799766
  T_ID_t1: 0.023529804220572
  T_FCNC_t1: 0.0207057808205794


## 7. Layer 5 — ERM Score Panel

In [8]:
dashboard["layer5_erm"] = {
    "national_erm_mean": clean(erm["ERM_final"].mean()),
    "erm_category_distribution": clean(erm["ERM_category"].value_counts().to_dict()),
    "erm_by_actor": clean(erm_actor.to_dict(orient="records")),
    "erm_by_stage": clean(erm_stage.to_dict(orient="records")),
}
print("Layer 5 - ERM per aktor:")
print(json.dumps(dashboard["layer5_erm"]["erm_by_actor"], indent=2, ensure_ascii=False))

Layer 5 - ERM per aktor:
[
  {
    "actor_category": "Judge",
    "mean": -0.027,
    "std": 0.42,
    "count": 182
  },
  {
    "actor_category": "Prosecutor",
    "mean": -0.026,
    "std": 0.386,
    "count": 179
  },
  {
    "actor_category": "CorrectionalOfficer",
    "mean": -0.022,
    "std": 0.424,
    "count": 198
  },
  {
    "actor_category": "Investigator",
    "mean": -0.017,
    "std": 0.402,
    "count": 196
  },
  {
    "actor_category": "Advocate",
    "mean": 0.005,
    "std": 0.389,
    "count": 127
  }
]


## 8. Simpan Dashboard JSON

In [9]:
dashboard = clean(dashboard)
with open("dashboard_data.json", "w") as f:
    json.dump(dashboard, f, indent=2, ensure_ascii=False)

# juga simpan versi flat per-responden utk dashboard yg butuh row-level
respondent_export = idx[["respondent_id","NREI_overall","NREI_overall_cat3"]] \
    .merge(erm[["respondent_id","ERM_final","ERM_category"]], on="respondent_id", how="left") \
    .merge(ml[["respondent_id","cluster_kmeans","anomaly_iso"]], on="respondent_id", how="left")
respondent_export.to_csv("dashboard_respondent_level.csv", index=False)

print("SELESAI.")
print(f"  dashboard_data.json ({len(dashboard)} layer)")
print(f"  dashboard_respondent_level.csv ({len(respondent_export)} responden)")

SELESAI.
  dashboard_data.json (6 layer)
  dashboard_respondent_level.csv (199 responden)


---
## Ringkasan File 07 & Keseluruhan Pipeline

**File 07** menghasilkan `dashboard_data.json` (struktur 5-layer siap-visualisasi) + `dashboard_respondent_level.csv` (data per responden).

### Pipeline lengkap yang sudah dibangun (File 00-07):

| File | Fungsi | Output utama |
|---|---|---|
| 00 | Data Cleaning | data_actor_clean, data_respondent_clean |
| 01 | Validitas & Reliabilitas | reliability_summary |
| 02 | Index Construction | data_index (MAS/ECI/NREI) |
| 03 | Sensitivity Analysis | flip-rate 6.8%, Spearman 0.99 |
| 04 | Feature Engineering | feature_matrix (70 kolom) |
| 05 | ML Analytics | cluster, klasifikasi, anomali |
| 06 | Explainability + ERM | SHAP/LIME, skor ERM |
| 07 | Dashboard Export | dashboard_data.json |

### Langkah berikutnya (di luar coding):
1. **Ganti data dummy dgn data riil** saat kuesioner terkumpul — seluruh pipeline tinggal dijalankan ulang, struktur kode tidak berubah.
2. **Expert panel** untuk bobot ERM final (Phase 2).
3. **Bangun visualisasi** dashboard dari `dashboard_data.json` (bisa pakai Streamlit/Dash/Power BI — di luar scope 8 notebook ini).
4. **Tinjau temuan sensitivity File 03** (flip-rate >5%) — putuskan apakah perlu penyesuaian bobot atau cukup didokumentasikan sebagai temuan Bab IV.

**Ingat batas etis (proposal §Ethical Boundaries):** seluruh output ini adalah alat bantu interpretasi berbasis persepsi dengan supervisi manusia (Human-in-the-Loop) — BUKAN bukti hukum, BUKAN alat memeringkat/menghukum aktor.